In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Generate synthetic X
num_subjects = 500
num_features = 5

X_raw = torch.randn(num_subjects, num_features)


# 2. Add intercept column
ones_column = torch.ones(num_subjects, 1)

X = torch.cat((ones_column, X_raw), dim=1)
# X shape: (500, 6)


# 3. Define true beta
# 第一個是 intercept
beta_true = torch.tensor([
    1.0,   # intercept
    2.0,   # X1 effect
    -1.5,  # X2 effect
    0.5,   # X3 effect
    0.0,   # X4 no effect
    3.0    # X5 effect
]).reshape(-1,1)


# 4. Generate Y according to linear model
noise = torch.randn(num_subjects,1) * 0.5

Y_raw = X @ beta_true + noise

In [2]:
# 確認模擬資料是對的
beta = torch.linalg.solve(
    X.T @ X,
    X.T @ Y_raw
)

print(beta)

tensor([[ 0.9520],
        [ 1.9933],
        [-1.4735],
        [ 0.4968],
        [ 0.0032],
        [ 3.0223]])


In [3]:
from sklearn.model_selection import train_test_split

# 建立 index
indices = torch.arange(num_subjects)

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42
)

# 切資料
X_train_raw = X_raw[train_idx]
X_test_raw = X_raw[test_idx]

Y_train = Y_raw[train_idx]
Y_test = Y_raw[test_idx]

In [4]:
# Train跟test都要加截距項
ones_train = torch.ones(X_train_raw.shape[0],1)
ones_test = torch.ones(X_test_raw.shape[0],1)


X_train = torch.cat(
    (ones_train, X_train_raw),
    dim=1
)

X_test = torch.cat(
    (ones_test, X_test_raw),
    dim=1
)

In [5]:
# 先把Y也加入X中，要讓attention matrix有Y的資訊
X_Y_train = torch.cat(
    (X_train, Y_train),
    dim=1
)

In [6]:
# 原始程式碼貼上(不訓練attention)

# Transpose data so the 6 features plus Y act as the "sequence" 
X_Y_features = X_Y_train.t()

# 5. Define the projection dimension (d_k)
d_k = 32
W_Q = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

W_K = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

# 6. Project features into Query (Q) and Key (K) spaces
Q = W_Q(X_Y_features)
K = W_K(X_Y_features)

# 7. Compute the raw attention scores
scores = torch.matmul(
    Q,
    K.transpose(-2,-1)
)

# 8. Scale by sqrt(d_k) and apply Softmax row-wise
attention_matrix = F.softmax(
    scores / (d_k ** 0.5),
    dim=-1
)

In [7]:
print("Attention Matrix Shape:", attention_matrix.shape)
print("\nAttention Matrix:\n", attention_matrix)

Attention Matrix Shape: torch.Size([7, 7])

Attention Matrix:
 tensor([[0.0989, 0.0983, 0.0788, 0.0771, 0.0651, 0.1284, 0.4535],
        [0.2939, 0.1125, 0.1179, 0.1040, 0.1420, 0.1360, 0.0938],
        [0.0744, 0.1622, 0.1266, 0.0562, 0.1054, 0.1374, 0.3379],
        [0.1426, 0.1134, 0.1718, 0.1821, 0.1195, 0.1525, 0.1181],
        [0.0586, 0.0433, 0.0416, 0.0350, 0.0493, 0.1053, 0.6668],
        [0.1241, 0.1590, 0.2121, 0.1383, 0.0900, 0.1625, 0.1139],
        [0.4933, 0.0376, 0.1752, 0.1158, 0.0194, 0.1426, 0.0162]],
       grad_fn=<SoftmaxBackward0>)


In [8]:
# 把Y再從矩陣中拿掉
A = attention_matrix[:-1,:-1]

print("Attention Matrix Shape(拿掉Y):", A.shape)
print("\nAttention Matrix(拿掉Y):\n", A)

Attention Matrix Shape(拿掉Y): torch.Size([6, 6])

Attention Matrix(拿掉Y):
 tensor([[0.0989, 0.0983, 0.0788, 0.0771, 0.0651, 0.1284],
        [0.2939, 0.1125, 0.1179, 0.1040, 0.1420, 0.1360],
        [0.0744, 0.1622, 0.1266, 0.0562, 0.1054, 0.1374],
        [0.1426, 0.1134, 0.1718, 0.1821, 0.1195, 0.1525],
        [0.0586, 0.0433, 0.0416, 0.0350, 0.0493, 0.1053],
        [0.1241, 0.1590, 0.2121, 0.1383, 0.0900, 0.1625]],
       grad_fn=<SliceBackward0>)


In [9]:
# 矩陣乘上Y的變異數
var_y = torch.var(Y_train)

A_var_y = A * var_y

In [10]:
A_var_y

tensor([[1.6217, 1.6131, 1.2921, 1.2642, 1.0680, 2.1062],
        [4.8211, 1.8452, 1.9337, 1.7053, 2.3293, 2.2315],
        [1.2201, 2.6604, 2.0764, 0.9212, 1.7297, 2.2540],
        [2.3394, 1.8607, 2.8187, 2.9866, 1.9598, 2.5022],
        [0.9620, 0.7102, 0.6831, 0.5744, 0.8087, 1.7281],
        [2.0358, 2.6090, 3.4799, 2.2686, 1.4767, 2.6666]],
       grad_fn=<MulBackward0>)

In [11]:
X_train.T @ X_train

tensor([[400.0000,   1.0350,  20.7889,  -0.7905,  13.4885, -24.8118],
        [  1.0350, 429.1736, -39.1551,  15.5268, -19.3454, -13.8929],
        [ 20.7889, -39.1551, 379.2039, -13.9947, -38.2809,  21.3498],
        [ -0.7905,  15.5268, -13.9947, 369.3360, -13.5115,  26.3914],
        [ 13.4885, -19.3454, -38.2809, -13.5115, 448.4792, -10.2828],
        [-24.8118, -13.8929,  21.3498,  26.3914, -10.2828, 423.5891]])

In [14]:
# 放入OLS的公式解中，估計beta
beta_attention = torch.linalg.solve(
    (X_train.T @ X_train + A_var_y)/2,
    X_train.T @ Y_train
)

print(
    "Attention beta:",
    beta_attention
)

Attention beta: tensor([[ 1.8757],
        [ 3.9088],
        [-3.0536],
        [ 0.9595],
        [-0.0626],
        [ 5.9731]], grad_fn=<LinalgSolveExBackward0>)


In [15]:
# 把算出來的係數套入驗證集
Y_pred_attention = X_test @ beta_attention

# 算MSE
mse_attention = torch.mean(
    (Y_test - Y_pred_attention)**2
)

print(
    "Attention MSE:",
    mse_attention.item()
)

Attention MSE: 15.819356918334961


In [16]:
# OLS的標準解法
beta_ols = torch.linalg.solve(
    X_train.T @ X_train,
    X_train.T @ Y_train
)

print(
    "OLS beta:",
    beta_ols
)

Y_pred_ols = X_test @ beta_ols

mse_ols = torch.mean(
    (Y_test - Y_pred_ols)**2
)

print(
    "OLS MSE:",
    mse_ols.item()
)

OLS beta: tensor([[ 0.9610],
        [ 1.9872],
        [-1.4960],
        [ 0.5066],
        [-0.0117],
        [ 3.0115]])
OLS MSE: 0.3120235502719879
